# July 1 Dipper ML Visualizations

Visual diagnostics for the July 1 `stats_only` dipper-like LightGBM model. The notebook reads saved model artifacts, candidate scores, feature-window summaries, and the review DB so it can be rerun after retraining.

In [ ]:
from pathlib import Path
import json
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_curve

from malca.io.lightcurve_io import load_lightcurve_df, stable_camera_color

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

def find_repo_root(start: Path) -> Path:
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "malca").is_dir() and (path / "output" / "runs").is_dir():
            return path
    raise FileNotFoundError("Could not locate the MALCA repo root from the current working directory")

REPO_ROOT = find_repo_root(Path.cwd())
RUN_DIR = REPO_ROOT / "output" / "runs" / "dat3-full-extended_2026-07-01-v4"
DB_PATH = RUN_DIR / "review" / "review.db"
BASE_DIR = Path("/tmp/malca_smplot_dipper_feature_selection")
MODEL_NAME = "stats_plus_periodicity_dip_jump"
MODEL_DIR = BASE_DIR / MODEL_NAME
FIG_DIR = REPO_ROOT / "output" / "pdf" / "smplotlib_copies_2026-07-22" / "ml_visualizations_smplotlib" / "dipper"
FIG_DIR.mkdir(parents=True, exist_ok=True)

POSITIVE_LABEL = "dipper_like"
PROB_COL = "prob_dipper_like"
POSITIVE_EVENT_CLASSES = {"dipper", "mixed_dip_and_burst"}
POSITIVE_MORPHOLOGIES = {"dimming_event", "mixed_dip_and_burst"}

assert MODEL_DIR.exists(), f"Model artifact directory not found: {MODEL_DIR}"

## Load Artifacts

In [ ]:
run_summary = json.loads((BASE_DIR / "run_summary.json").read_text())
metadata = json.loads((MODEL_DIR / "metadata.json").read_text())
label_audit = json.loads((BASE_DIR / "label_audit.json").read_text())

importance = pd.read_csv(MODEL_DIR / "feature_importance_gain.csv")
windows = pd.read_csv(MODEL_DIR / "feature_windows.csv")
scores = pd.read_parquet(MODEL_DIR / "all_candidates_scores.parquet")
queue = pd.read_csv(MODEL_DIR / "high_priority_review_queue.csv")
test_predictions = pd.read_parquet(MODEL_DIR / "test_predictions.parquet")
confusion = pd.read_csv(MODEL_DIR / "confusion_matrix.csv")
calibration = pd.read_csv(MODEL_DIR / "calibration_by_bin.csv")
cv_metrics = pd.read_csv(MODEL_DIR / "cv_metrics.csv")

model_summary = run_summary["models"][0]
summary_text = (
    f"**Model:** `{MODEL_NAME}`  \\n"
    f"**Training rows:** {model_summary['n_trainable_rows']:,}  \\n"
    f"**Features:** {model_summary['n_features']:,}  \\n"
    f"**Class counts:** {model_summary['class_counts']}  \\n"
    f"**Scores:** {len(scores):,} candidates"
)
display(Markdown(summary_text))
display(pd.DataFrame(label_audit["target_counts"].items(), columns=["target", "count"]))
display(cv_metrics)

## Held-Out Performance

In [ ]:
cm = confusion.set_index("y_true")
fig, ax = plt.subplots(figsize=(5.5, 4.8))
image = ax.imshow(cm.values, cmap="Blues")
ax.set_xticks(range(len(cm.columns)), labels=cm.columns, rotation=30, ha="right")
ax.set_yticks(range(len(cm.index)), labels=cm.index)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Held-out confusion matrix")
for row_idx in range(cm.shape[0]):
    for col_idx in range(cm.shape[1]):
        value = int(cm.iloc[row_idx, col_idx])
        color = "white" if value > cm.values.max() / 2 else "black"
        ax.text(col_idx, row_idx, str(value), ha="center", va="center", color=color, fontsize=12)
fig.colorbar(image, ax=ax, shrink=0.8, label="Candidates")
fig.tight_layout()
fig.savefig(FIG_DIR / "confusion_matrix.png", dpi=180)
plt.show()

In [ ]:
y_true = test_predictions["y_true"].eq(POSITIVE_LABEL).astype(int)
y_score = test_predictions[PROB_COL]

fpr, tpr, _ = roc_curve(y_true, y_score)
precision, recall, _ = precision_recall_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
pr_auc = average_precision_score(y_true, y_score)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], color="0.6", lw=1, linestyle="--")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curve")
axes[0].legend(loc="lower right")

baseline = y_true.mean()
axes[1].plot(recall, precision, lw=2, label=f"AP = {pr_auc:.3f}")
axes[1].axhline(baseline, color="0.6", lw=1, linestyle="--", label=f"Baseline = {baseline:.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-recall curve")
axes[1].legend(loc="upper right")

fig.tight_layout()
fig.savefig(FIG_DIR / "roc_pr_curves.png", dpi=180)
plt.show()

In [ ]:
cal = calibration.loc[calibration["class_label"].eq(POSITIVE_LABEL)].copy()
cal = cal.dropna(subset=["mean_probability", "observed_rate"])

fig, ax = plt.subplots(figsize=(5.5, 5))
if not cal.empty:
    sizes = 30 + 6 * cal["n"].astype(float)
    ax.scatter(cal["mean_probability"], cal["observed_rate"], s=sizes, alpha=0.75, edgecolor="black")
ax.plot([0, 1], [0, 1], color="0.5", linestyle="--", lw=1)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed dipper-like rate")
ax.set_title("Held-out calibration by probability bin")
fig.tight_layout()
fig.savefig(FIG_DIR / "calibration_dipper_like.png", dpi=180)
plt.show()
display(cal)

## Feature Importance

In [ ]:
top_n = 25
top_importance = importance.head(top_n).iloc[::-1]

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(top_importance["feature"], top_importance["gain"], color="#315f72")
ax.set_xlabel("Feature importance")
fig.tight_layout()
fig.savefig(FIG_DIR / "top_feature_importance_gain.png", dpi=180)
plt.show()
display(importance.head(30))

## Candidate Score Distribution

In [ ]:
scores = scores.copy()
event_class = scores.get("event_class", pd.Series("", index=scores.index)).fillna("").astype(str).str.strip()
workflow = scores.get("workflow_status", pd.Series("", index=scores.index)).fillna("").astype(str).str.strip()
scores["score_group"] = np.select(
    [
        event_class.isin(POSITIVE_EVENT_CLASSES),
        workflow.ne("") & workflow.ne("unreviewed"),
    ],
    ["reviewed_dipper_like", "reviewed_other"],
    default="unreviewed",
)

fig, ax = plt.subplots(figsize=(9, 5))
bins = np.r_[np.linspace(0, 0.02, 30), np.linspace(0.025, 1.0, 45)]
for group, color in [
    ("unreviewed", "#9aa0a6"),
    ("reviewed_other", "#c15f5f"),
    ("reviewed_dipper_like", "#315f72"),
]:
    values = scores.loc[scores["score_group"].eq(group), PROB_COL].dropna()
    if values.empty:
        continue
    weights = np.ones(len(values), dtype=float) / len(values)
    ax.hist(values, bins=bins, weights=weights, histtype="step", lw=2, label=f"{group} (n={len(values):,})", color=color)
ax.set_yscale("log")
ax.set_xlabel("Predicted dipper-like probability")
ax.set_ylabel("Fraction of group")
ax.set_title("Score distribution across reviewed and unreviewed candidates")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "score_distribution.png", dpi=180)
plt.show()

display(scores[[PROB_COL]].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
display(pd.cut(scores[PROB_COL], [0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1], include_lowest=True).value_counts().sort_index())

In [ ]:
TOP_PROBABILITY_N = 500

top_probability = queue.sort_values(PROB_COL, ascending=False).head(TOP_PROBABILITY_N).reset_index(drop=True)
top_probability["rank_by_dipper_probability"] = np.arange(1, len(top_probability) + 1)

fig, ax = plt.subplots(figsize=(11, 5.5))
scatter = ax.scatter(
    top_probability["rank_by_dipper_probability"],
    top_probability[PROB_COL],
    c=pd.to_numeric(top_probability["dipper_score"], errors="coerce"),
    cmap="magma",
    s=34,
    edgecolor="black",
    linewidth=0.25,
)
ax.plot(
    top_probability["rank_by_dipper_probability"],
    top_probability[PROB_COL],
    color="0.55",
    lw=1,
    alpha=0.65,
)
ax.set_xlabel("Rank among unreviewed candidates by predicted dipper-like probability")
ax.set_ylabel("Predicted dipper-like probability")
ax.set_title(f"Top {TOP_PROBABILITY_N} unreviewed candidates by predicted dipper-like probability")
colorbar = fig.colorbar(scatter, ax=ax, shrink=0.85)
colorbar.set_label("dipper_score")
fig.tight_layout()
fig.savefig(FIG_DIR / f"top{TOP_PROBABILITY_N}_unreviewed_by_dipper_probability.png", dpi=180)
plt.show()

top_probability_columns = [
    "rank_by_dipper_probability",
    "candidate_id",
    "asas_sn_id",
    PROB_COL,
    "dipper_score",
    "dipper_n_dips",
    "dipper_n_valid_dips",
    "dip_significant",
    "vetting_likely_known",
    "catalog_match",
    "catalog_source",
    "simbad_otype",
    "vsx_class",
]
top_probability_view = top_probability[[column for column in top_probability_columns if column in top_probability.columns]].copy()
top_probability_view.to_csv(MODEL_DIR / f"top{TOP_PROBABILITY_N}_unreviewed_by_dipper_probability.csv", index=False)
display(top_probability_view)


## Top 500 Probability-Ranked Light Curves

In [ ]:
LIGHTCURVE_DIR = RUN_DIR / "bundle_assets" / "lightcurves"
LC_GRID_DIR = FIG_DIR / f"top{TOP_PROBABILITY_N}_probability_lightcurves"
LC_GRID_DIR.mkdir(parents=True, exist_ok=True)


def resolve_bundle_lightcurve(row: pd.Series) -> Path | None:
    stems = []
    for column in ("asas_sn_id", "candidate_id"):
        if column not in row or pd.isna(row[column]):
            continue
        value = str(row[column]).strip()
        if not value:
            continue
        stems.append(value.removeprefix("stv_"))
        stems.append(value)
    for stem in dict.fromkeys(stems):
        for suffix in (".dat3", ".dat2", ".dat"):
            path = LIGHTCURVE_DIR / f"{stem}{suffix}"
            if path.exists():
                return path
    return None


def plot_lightcurve_panel(ax, row: pd.Series) -> None:
    path = resolve_bundle_lightcurve(row)
    title = f"{int(row['rank_by_dipper_probability']):03d} {row['candidate_id']}"
    subtitle = f"p={row[PROB_COL]:.3f}, score={pd.to_numeric(row.get('dipper_score'), errors='coerce'):.2f}"
    if path is None:
        ax.text(0.5, 0.5, "missing light curve", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{title}\n{subtitle}", fontsize=8)
        ax.set_axis_off()
        return
    try:
        lc = load_lightcurve_df(path, apply_quality=True)
    except Exception as exc:
        ax.text(0.5, 0.5, f"load failed\n{exc}", ha="center", va="center", fontsize=7, transform=ax.transAxes)
        ax.set_title(f"{title}\n{subtitle}", fontsize=8)
        ax.set_axis_off()
        return
    if lc.empty:
        ax.text(0.5, 0.5, "empty light curve", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{title}\n{subtitle}", fontsize=8)
        ax.set_axis_off()
        return
    jd = pd.to_numeric(lc["jd"], errors="coerce")
    mag = pd.to_numeric(lc["mag"], errors="coerce")
    camera = lc.get("camera_name", lc.get("camera", pd.Series("all", index=lc.index))).fillna("all").astype(str)
    band = lc.get("band", pd.Series("", index=lc.index)).fillna("").astype(str).str.lower()
    finite = jd.notna() & mag.notna()
    if not bool(finite.any()):
        ax.text(0.5, 0.5, "no finite photometry", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{title}\n{subtitle}", fontsize=8)
        ax.set_axis_off()
        return
    for cam, idx in camera.loc[finite].groupby(camera.loc[finite]).groups.items():
        idx = pd.Index(idx)
        color = stable_camera_color(str(cam))
        marker = "s" if band.loc[idx].eq("v").any() and not band.loc[idx].eq("g").any() else "o"
        ax.scatter(
            jd.loc[idx] - 2450000.0,
            mag.loc[idx],
            marker=marker,
            s=5.0,
            alpha=0.72,
            color=color,
            linewidths=0,
        )
    ax.invert_yaxis()
    ax.set_title(f"{title}\n{subtitle}", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.25)


top_probability = queue.sort_values(PROB_COL, ascending=False).head(TOP_PROBABILITY_N).reset_index(drop=True)
top_probability["rank_by_dipper_probability"] = np.arange(1, len(top_probability) + 1)
top_probability["resolved_lightcurve_path"] = top_probability.apply(resolve_bundle_lightcurve, axis=1).map(lambda p: str(p) if p else "")
top_probability.to_csv(MODEL_DIR / f"top{TOP_PROBABILITY_N}_unreviewed_by_dipper_probability_with_lightcurve_paths.csv", index=False)

page_paths = []
per_page = 20
for page_idx, start in enumerate(range(0, len(top_probability), per_page), start=1):
    page = top_probability.iloc[start:start + per_page]
    fig, axes = plt.subplots(5, 4, figsize=(16, 13), sharex=False, sharey=False)
    axes = axes.ravel()
    for ax, (_, row) in zip(axes, page.iterrows()):
        plot_lightcurve_panel(ax, row)
    for ax in axes[len(page):]:
        ax.set_axis_off()
    fig.supxlabel("JD - 2450000", fontsize=11)
    fig.supylabel("ASAS-SN mag", fontsize=11)
    fig.suptitle(f"Top {TOP_PROBABILITY_N} unreviewed dipper candidates, pg. {page_idx}", y=0.995, fontsize=14)
    fig.tight_layout(rect=[0.02, 0.02, 1.0, 0.975])
    page_path = LC_GRID_DIR / f"top{TOP_PROBABILITY_N}_probability_lightcurves_page{page_idx:02d}.png"
    fig.savefig(page_path, dpi=180)
    page_paths.append(page_path)
    plt.show()

display(Markdown("Saved light-curve grid pages:\n" + "\n".join(f"- `{path}`" for path in page_paths)))
display(top_probability[["rank_by_dipper_probability", "candidate_id", PROB_COL, "dipper_score", "resolved_lightcurve_path"]].head(20))


## Highest Dipper-Score Unreviewed Candidates

In [ ]:
workflow = scores.get("workflow_status", pd.Series("", index=scores.index)).fillna("").astype(str).str.strip()
event_class = scores.get("event_class", pd.Series("", index=scores.index)).fillna("").astype(str).str.strip()
unreviewed_mask = workflow.isin(("", "unreviewed")) & event_class.isin(("", "unclassified"))

top100_dipper_score = scores.loc[unreviewed_mask].copy()
top100_dipper_score["dipper_score_num"] = pd.to_numeric(top100_dipper_score["dipper_score"], errors="coerce")
top100_dipper_score = (
    top100_dipper_score.dropna(subset=["dipper_score_num"])
    .sort_values(["dipper_score_num", PROB_COL], ascending=[False, False])
    .head(100)
    .reset_index(drop=True)
)
top100_dipper_score["rank_by_dipper_score"] = np.arange(1, len(top100_dipper_score) + 1)

fig, ax = plt.subplots(figsize=(11, 5.5))
scatter = ax.scatter(
    top100_dipper_score["rank_by_dipper_score"],
    top100_dipper_score["dipper_score_num"],
    c=top100_dipper_score[PROB_COL],
    cmap="viridis",
    s=42,
    edgecolor="black",
    linewidth=0.35,
)
ax.plot(
    top100_dipper_score["rank_by_dipper_score"],
    top100_dipper_score["dipper_score_num"],
    color="0.55",
    lw=1,
    alpha=0.65,
)
ax.set_xlabel("Rank among unreviewed candidates by dipper_score")
ax.set_ylabel("dipper_score")
ax.set_title("Top 100 unreviewed candidates by dipper_score")
colorbar = fig.colorbar(scatter, ax=ax, shrink=0.85)
colorbar.set_label("Predicted dipper-like probability")
fig.tight_layout()
fig.savefig(FIG_DIR / "top100_unreviewed_by_dipper_score.png", dpi=180)
plt.show()

top100_columns = [
    "rank_by_dipper_score",
    "candidate_id",
    "asas_sn_id",
    "dipper_score",
    PROB_COL,
    "dipper_n_dips",
    "dipper_n_valid_dips",
    "dip_significant",
    "vetting_likely_known",
    "catalog_match",
    "catalog_source",
    "simbad_otype",
    "vsx_class",
]
top100_view = top100_dipper_score[[column for column in top100_columns if column in top100_dipper_score.columns]].copy()
top100_view.to_csv(MODEL_DIR / "top100_unreviewed_by_dipper_score.csv", index=False)
display(top100_view)

## Deterministic Feature Windows

## Reviewed Dipper Candidates

In [ ]:
workflow = scores.get("workflow_status", pd.Series("", index=scores.index)).fillna("").astype(str).str.strip()
event_class = scores.get("event_class", pd.Series("", index=scores.index)).fillna("").astype(str).str.strip()
morphology = scores.get("morphology_primary", pd.Series("", index=scores.index)).fillna("").astype(str).str.strip()
reviewed_mask = workflow.ne("") & workflow.ne("unreviewed")
reviewed_dipper_mask = reviewed_mask & (event_class.isin(POSITIVE_EVENT_CLASSES) | morphology.isin(POSITIVE_MORPHOLOGIES))

reviewed_dippers = scores.loc[reviewed_dipper_mask].copy()
reviewed_dippers["dipper_score_num"] = pd.to_numeric(reviewed_dippers["dipper_score"], errors="coerce")
reviewed_dippers = (
    reviewed_dippers.sort_values(["dipper_score_num", PROB_COL], ascending=[False, False], na_position="last")
    .reset_index(drop=True)
)
reviewed_dippers["rank_by_dipper_score"] = np.arange(1, len(reviewed_dippers) + 1)

fig, ax = plt.subplots(figsize=(11, 5.5))
scatter = ax.scatter(
    reviewed_dippers["rank_by_dipper_score"],
    reviewed_dippers["dipper_score_num"],
    c=reviewed_dippers[PROB_COL],
    cmap="viridis",
    s=48,
    edgecolor="black",
    linewidth=0.35,
)
ax.axhline(0, color="0.55", lw=1, linestyle="--")
ax.set_xlabel("Rank among reviewed dipper-like candidates by dipper_score")
ax.set_ylabel("dipper_score")
ax.set_title(f"Reviewed dipper-like candidates by dipper_score (n={len(reviewed_dippers):,})")
colorbar = fig.colorbar(scatter, ax=ax, shrink=0.85)
colorbar.set_label("Predicted dipper-like probability")
fig.tight_layout()
fig.savefig(FIG_DIR / "reviewed_dippers_by_dipper_score.png", dpi=180)
plt.show()

reviewed_dipper_columns = [
    "rank_by_dipper_score",
    "candidate_id",
    "asas_sn_id",
    "event_class",
    "morphology_primary",
    "workflow_status",
    "dipper_score",
    PROB_COL,
    "dipper_n_dips",
    "dipper_n_valid_dips",
    "dip_significant",
    "vetting_likely_known",
    "catalog_match",
    "catalog_source",
    "simbad_otype",
    "vsx_class",
]
reviewed_dipper_view = reviewed_dippers[[column for column in reviewed_dipper_columns if column in reviewed_dippers.columns]].copy()
reviewed_dipper_view.to_csv(MODEL_DIR / "reviewed_dippers_by_dipper_score.csv", index=False)
display(reviewed_dipper_view)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for coverage, color in [(1.0, "#444444"), (0.95, "#5a8f5b"), (0.90, "#315f72"), (0.80, "#c27c3a")]:
    subset = windows.loc[windows["target_coverage"].eq(coverage)].copy()
    ax.scatter(
        subset["all_candidate_fraction_kept"],
        subset["reviewed_precision"],
        s=30 + 80 * subset["positive_recall"].fillna(0),
        alpha=0.65,
        label=f"{coverage:.0%} dipper window",
        color=color,
    )
ax.set_xlabel("Fraction of all candidates kept by one-feature window")
ax.set_ylabel("Reviewed precision inside window")
ax.set_title("One-feature window tradeoffs")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "feature_window_tradeoffs.png", dpi=180)
plt.show()

best_90 = (
    windows.loc[windows["target_coverage"].eq(0.90)]
    .sort_values(["positive_recall", "all_candidate_fraction_kept", "reviewed_precision"], ascending=[False, True, False])
    .head(30)
)
display(best_90)

## Top Feature Distributions

In [ ]:
def quote_identifier(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'

def read_candidate_columns(db_path: Path, columns: list[str]) -> pd.DataFrame:
    select_list = ", ".join(quote_identifier(column) for column in columns)
    with sqlite3.connect(f"file:{db_path.resolve()}?mode=ro", uri=True) as conn:
        return pd.read_sql_query(f"SELECT {select_list} FROM candidates", conn)

top_features = importance["feature"].head(8).tolist()
feature_values = read_candidate_columns(DB_PATH, ["candidate_id", *top_features])
feature_values["candidate_id"] = feature_values["candidate_id"].astype(str)

dist = scores[["candidate_id", "event_class", "morphology_primary", "workflow_status", PROB_COL]].merge(
    feature_values,
    on="candidate_id",
    how="left",
)
event_class = dist["event_class"].fillna("").astype(str).str.strip()
morphology = dist["morphology_primary"].fillna("").astype(str).str.strip()
workflow = dist["workflow_status"].fillna("").astype(str).str.strip()
dist["label_group"] = np.select(
    [
        event_class.isin(POSITIVE_EVENT_CLASSES) | morphology.isin(POSITIVE_MORPHOLOGIES),
        workflow.ne("") & workflow.ne("unreviewed"),
    ],
    ["reviewed_dipper_like", "reviewed_other"],
    default="unreviewed",
)

n_cols = 2
n_rows = int(np.ceil(len(top_features) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3.2 * n_rows))
axes = np.asarray(axes).ravel()
for ax, feature in zip(axes, top_features):
    values = pd.to_numeric(dist[feature], errors="coerce").replace([np.inf, -np.inf], np.nan)
    finite = values.dropna()
    if finite.empty:
        ax.set_axis_off()
        continue
    lo, hi = finite.quantile([0.01, 0.99])
    bins = np.linspace(lo, hi, 35) if np.isfinite(lo) and np.isfinite(hi) and lo < hi else 20
    for group, color, alpha in [
        ("reviewed_other", "#c15f5f", 0.35),
        ("unreviewed", "#9aa0a6", 0.25),
        ("reviewed_dipper_like", "#315f72", 0.55),
    ]:
        group_values = values.loc[dist["label_group"].eq(group)].dropna()
        group_values = group_values.loc[group_values.between(lo, hi, inclusive="both")]
        if group_values.empty:
            continue
        weights = np.ones(len(group_values), dtype=float) / len(group_values)
        ax.hist(group_values, bins=bins, weights=weights, alpha=alpha, label=group, color=color)
    ax.set_title(feature)
    ax.set_ylabel("Fraction of group")
for ax in axes[len(top_features):]:
    ax.set_axis_off()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(FIG_DIR / "top_feature_distributions.png", dpi=180)
plt.show()

In [ ]:
corr_cols = [PROB_COL, *top_features]
corr = dist[corr_cols].apply(pd.to_numeric, errors="coerce").corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(corr.values, cmap="vlag" if "vlag" in plt.colormaps() else "coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), labels=corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)), labels=corr.index)
ax.set_title("Spearman correlation among score and top features")
fig.colorbar(image, ax=ax, shrink=0.8, label="Spearman rho")
fig.tight_layout()
fig.savefig(FIG_DIR / "score_top_feature_correlation.png", dpi=180)
plt.show()
display(corr)

## Next Checks

Use `high_priority_review_queue.csv` for visual review first. Treat high predicted probability as a ranking signal, not as an automatic dipper label. The one-feature windows are useful for turning the learned model back into deterministic candidate cuts, but their contamination rates should be checked before any review is skipped.